# BP6 Gate 3 — Retrieval Strategy Benchmark & Champion Selection
**Customer360 Navigator Enterprise Suite — GenAI Resolution Assistant**

## Why this gate looks different from BP1-5's own Gate 3

The Master Plan's generic Gate 3 row ("Model / Classifier Benchmark & Champion Selection" —
output: "Benchmarked model set, champion selected"; exit criteria: "Identical CV folds across
candidates; champion picked by mean CV metric"; compliance touchpoint: "Model inventory entry
opened (SR 11-7 first-line record)") assumes a supervised classifier. **BP6 has none, at any
gate.** The real thing BP6 needs benchmarked before its Gate 5 retrieval-and-generation step is a
**retrieval strategy**: a function that, given a query on one side of the CFPB<->BANKING77 divide,
finds real evidence on the other side. This gate benchmarks two real, disclosed candidates:

- **`taxonomy_bucket_match`** — query -> `common_taxonomy_bucket` (via BP1's own real,
  already-built crosswalk, `configs/taxonomy_mapping.yaml` / `src/taxonomy/taxonomy_mapper.py`,
  reused unmodified) -> retrieve every real record on the other side sharing that bucket.
- **`raw_string_match`** (naive baseline) — retrieve records whose raw Product/category string is
  byte-identical to the query, with no crosswalk at all.

Both are scored by a real, structurally-computed **coverage** metric (no ML training, no
synthetic ground truth — there is no labeled "correct retrieval" dataset to validate against; the
correctness signal is structural: does a query's real bucket actually have matching records on the
other side). `taxonomy_bucket_match` wins for real, disclosed reasons: CFPB and BANKING77 share
zero columns and zero overlapping raw vocabulary (RAW_DATA_MANIFEST.md Finding 2, re-verified at
every gate that has checked it), so `raw_string_match` is expected — and confirmed below — to
score at or near zero. This is not a foregone conclusion dressed up: `taxonomy_bucket_match`'s own
coverage is genuinely imperfect (real buckets on the BANKING77 side legitimately have no CFPB-side
counterpart, since only ~6.55% of the real CFPB extract has any BANKING77 overlap at all, per
`CFPB_BANKING77_TAXONOMY_MAPPING.md`), and that imperfection is reported, not hidden.

## What this gate does

1. **Real bucket counts** (Section 4) — reads the `common_taxonomy_bucket` column directly off
   BP1's own real, already-built Gold layers on both sides (`banking77_common_taxonomy_gold.parquet`,
   `cfpb_common_taxonomy_gold.parquet`) — never a fresh scan of the raw CFPB/BANKING77 files.
2. **Benchmark** (Section 5) — `taxonomy_bucket_match_coverage` / `raw_string_match_coverage`
   (`src/genai/bp6_retrieval_candidates.py`, new module) computed over every real bucket/product
   value actually present, not a synthetic query set.
3. **Champion selection** (Section 5) — deterministic, higher real coverage wins; a tie is
   disclosed explicitly rather than silently broken.
4. **Zero-GenAI-SDK-loaded re-check** (Section 6) — same pattern and prefix list as Gates 1-2.
5. **Retrieval strategy inventory entry** (Section 7) — BP6's real analog of Gate 3's own "Model
   inventory entry opened (SR 11-7 first-line record)" compliance touchpoint, shaped after BP1-5's
   own real `model_inventory_entry.json` (champion/runner-up/candidates_evaluated/real metrics),
   with field names adapted since BP6's "champion" is a retrieval strategy, never a model.
6. **Config write** (Section 8) — a nested `gate3_retrieval_benchmark:` dict block, matching
   BP1-5's own convention that nested gate blocks start at Gate 3 (Gate 2 is flat keys across this
   whole project).
7. **Structural integrity checks** (Section 9) — 10 named assertions, including that the naive
   baseline is structurally confirmed weaker than the taxonomy-bucket strategy (not just assumed).

## No bugs caught this gate

Unlike Gates 6/7 of BP5 and Gate 2 of BP6, this gate's own sandbox verification did not surface a
defect before delivery — first-pass real numbers (bucket counts, coverage, champion selection)
matched hand-derived expectations exactly, and the idempotency re-run produced identical config
values.

## Prerequisite

BP6 Gate 2 must have already run for real (writes this config's Gate 2 block), and BP1 Gate 2 must
have already run for real (writes both real Gold layers this notebook reads). Both are already
real-run confirmed as of this gate's delivery. Per this project's standing execution-boundary
rule, Claude never runs this notebook — only the user does, in the `home_credit_env` Jupyter
kernel.

## What this gate does NOT do

- **No re-derivation of the taxonomy crosswalk.** `configs/taxonomy_mapping.yaml` and
  `src/taxonomy/taxonomy_mapper.py` are reused exactly as BP1 Gate 2 built them.
- **No ML training, no CV folds, no synthetic ground truth.** There is no supervised target here
  to train or validate against — coverage is a real, structural fact read off the actual data.
- **No external call of any kind.** BP6's own GenAI call does not occur until Gate 5.
- **No financial-impact or illustrative-projection content, anywhere.**

## Real, disclosed design choices in this gate

- The naive `raw_string_match` baseline is a genuine benchmark, not a strawman inserted only to
  lose — it is exactly what BP6's retrieval would look like without the taxonomy-bucket mechanism,
  and its real, near-zero coverage is the honest empirical reason that mechanism is used.
- Coverage counts every real bucket value present on EITHER side, not only the buckets with the
  most rows — a rare BANKING77-side bucket with a real CFPB-side counterpart counts the same as a
  common one, so this metric reports reach, not just volume.

Every real number in this gate is read directly from BP1's own real, already real-run-confirmed
Gold layers, `configs/taxonomy_mapping.yaml`, and `data/external/banking77_categories.json`.
Nothing is estimated, assumed, or synthesized.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP6 Gate 3 (Retrieval Strategy Benchmark & Champion
Selection) notebook. Single consolidated code cell (platform convention). Idempotent.

BP6 has no supervised target and no classifier at any gate. This notebook maps the Master Plan's
generic Gate 3 row onto BP6's real shape: the "model" BP6 needs before its Gate 5 retrieval-and-
generation step is a retrieval strategy for linking a query context on one side of the CFPB<->
BANKING77 divide to real evidence on the other. Two real, disclosed candidate strategies are
benchmarked (src/genai/bp6_retrieval_candidates.py) and a champion selected by real, structurally
-computed coverage - no ML training, no synthetic ground truth.
"""

import os, sys, json, warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (same resolver as every other notebook)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)
    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb

perf_summary = configure_performance(project_root=PROJECT_ROOT)
print(f"[WARP] Headroom before work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Heavy imports + dependency/prerequisite check
# ============================================================
from taxonomy.taxonomy_mapper import load_mapping_config
from genai.bp6_retrieval_candidates import (
    load_taxonomy_linked_bucket_counts,
    taxonomy_bucket_match_coverage,
    raw_string_match_coverage,
    select_champion_retrieval_strategy,
)
from genai.bp6_evidence_prep import genai_sdk_modules_loaded

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EXTERNAL = PROJECT_ROOT / "data" / "external"
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp6_genai_resolution_assistant" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

BP6_CONFIG_PATH = CONFIGS_DIR / "bp6_genai_resolution_assistant.yaml"
TAXONOMY_MAPPING_PATH = CONFIGS_DIR / "taxonomy_mapping.yaml"
B77_GOLD_PATH = DATA_PROCESSED / "banking77_common_taxonomy_gold.parquet"
CFPB_GOLD_PATH = DATA_PROCESSED / "cfpb_common_taxonomy_gold.parquet"
B77_CATEGORIES_PATH = DATA_EXTERNAL / "banking77_categories.json"
GATE2_MARKER_TEXT = "Gate 2 (Data Verification & Feature/Taxonomy Engineering)"

for p in (BP6_CONFIG_PATH, TAXONOMY_MAPPING_PATH, B77_GOLD_PATH, CFPB_GOLD_PATH, B77_CATEGORIES_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} does not exist. Prerequisite: BP6 Gate 2 must have run for real at least once, "
            "and BP1 Gate 2 must have run for real at least once (writes both real Gold layers "
            "this notebook reads)."
        )
if GATE2_MARKER_TEXT not in BP6_CONFIG_PATH.read_text(encoding="utf-8"):
    raise RuntimeError("BP6 Gate 2's own config block was not found - run BP6 Gate 2 first.")
print("[OK] Prerequisites confirmed: BP6 Gate 2 config block present, both real Gold layers exist.")

# ============================================================
# SECTION 4: Load real per-bucket counts from BOTH real Gold layers (never re-derived, never a
# fresh scan of the raw CFPB/BANKING77 files - reads only the common_taxonomy_bucket column each
# Gold layer already carries, via BP1's own real, already-built crosswalk).
# ============================================================
mapping = load_mapping_config(TAXONOMY_MAPPING_PATH)
b77_bucket_counts, cfpb_bucket_counts = load_taxonomy_linked_bucket_counts(B77_GOLD_PATH, CFPB_GOLD_PATH)
print(
    f"[OK] Real bucket counts loaded: {b77_bucket_counts.height} distinct buckets on the "
    f"BANKING77 side, {cfpb_bucket_counts.height} on the CFPB side."
)

with open(B77_CATEGORIES_PATH, "r", encoding="utf-8") as f:
    b77_categories = json.load(f)
assert len(b77_categories) == 77, f"Expected 77 real BANKING77 categories, found {len(b77_categories)}"

# ============================================================
# SECTION 5: Benchmark both real candidate retrieval strategies
# ============================================================
strategy_a = taxonomy_bucket_match_coverage(b77_bucket_counts, cfpb_bucket_counts)
strategy_b = raw_string_match_coverage(mapping, b77_categories)
print(
    f"[OK] Strategy A (taxonomy_bucket_match) real coverage: {strategy_a['coverage']:.4f} "
    f"({strategy_a['n_buckets_with_cross_corpus_hit']}/{strategy_a['n_query_buckets']} real "
    "buckets have a cross-corpus hit)."
)
print(
    f"[OK] Strategy B (raw_string_match, naive baseline) real coverage: {strategy_b['coverage']:.4f} "
    f"({strategy_b['n_products_with_raw_string_hit']}/{strategy_b['n_query_products']} real CFPB "
    "products match a raw BANKING77 category string)."
)

champion_result = select_champion_retrieval_strategy(strategy_a, strategy_b)
print(f"[OK] Champion retrieval strategy selected: {champion_result['champion_strategy']} "
      f"(coverage {champion_result['champion_coverage']:.4f} vs runner-up "
      f"{champion_result['runner_up_strategy']} at {champion_result['runner_up_coverage']:.4f}).")

champion_is_taxonomy_bucket_match = champion_result["champion_strategy"] == "taxonomy_bucket_match"

# ============================================================
# SECTION 6: Zero-GenAI-SDK-loaded re-check (same pattern as Gates 1 and 2)
# ============================================================
loaded_sdks = genai_sdk_modules_loaded()
print(f"[OK] GenAI SDK modules loaded in this run: {loaded_sdks or 'NONE'}")

# ============================================================
# SECTION 7: Write the retrieval-strategy inventory entry - BP6's real analog of Gate 3's own
# "Model inventory entry opened (SR 11-7 first-line record)" compliance touchpoint, mirroring
# BP1-5's own real model_inventory_entry.json shape (candidates_evaluated / champion / real
# metrics), adapted field names since BP6's "champion" is a retrieval strategy, never a model.
# ============================================================
generated_at_utc = datetime.now(timezone.utc).isoformat()

retrieval_strategy_inventory_entry = {
    "bp_id": "bp6",
    "gate": 3,
    "compliance_touchpoint": "Retrieval strategy inventory entry opened (SR 11-7 first-line "
    "record analog - BP6 has no classifier, so the entry records a retrieval strategy, not a "
    "model)",
    "champion_strategy": champion_result["champion_strategy"],
    "champion_coverage": champion_result["champion_coverage"],
    "runner_up_strategy": champion_result["runner_up_strategy"],
    "runner_up_coverage": champion_result["runner_up_coverage"],
    "candidates_evaluated": champion_result["candidates_evaluated"],
    "candidates_failed": [],
    "strategy_a_detail": {k: v for k, v in strategy_a.items() if k not in ("query_buckets", "buckets_with_hit")},
    "strategy_b_detail": {k: v for k, v in strategy_b.items() if k not in ("query_products", "products_with_hit")},
    "generated_at_utc": generated_at_utc,
}
RETRIEVAL_INVENTORY_PATH = ARTIFACTS_DIR / "gate3_retrieval_strategy_inventory_entry.json"
with open(RETRIEVAL_INVENTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(retrieval_strategy_inventory_entry, f, indent=2)
print(f"[SAVED] Retrieval strategy inventory entry: {RETRIEVAL_INVENTORY_PATH}")

# ============================================================
# SECTION 8: Write the Gate 3 config block. Nested dict block (gate3_retrieval_benchmark),
# matching BP1's/BP2's/BP3's/BP4's/BP5's own established convention that nested blocks start at
# Gate 3 (Gate 2's own convention across this whole project is flat top-level keys).
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

gate3_marker = (
    "# --- Gate 3 (Retrieval Strategy Benchmark & Champion Selection) results "
    "(appended, idempotent overwrite) ---"
)
gate3_block_lines = [
    "gate3_retrieval_benchmark:",
    f'  champion_strategy: "{champion_result["champion_strategy"]}"',
    f'  champion_coverage: {champion_result["champion_coverage"]}',
    f'  runner_up_strategy: "{champion_result["runner_up_strategy"]}"',
    f'  runner_up_coverage: {champion_result["runner_up_coverage"]}',
    f"  candidates_evaluated: {champion_result['candidates_evaluated']}",
    f'  retrieval_strategy_inventory_entry_path: "{RETRIEVAL_INVENTORY_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f"  genai_sdk_modules_loaded_this_run: {loaded_sdks}",
    f'  generated_at_utc: "{generated_at_utc}"',
]
write_gate_block(BP6_CONFIG_PATH, gate3_marker, gate3_block_lines)
print(f"[SAVED] Gate 3 config block written to {BP6_CONFIG_PATH}")

# ============================================================
# SECTION 9: Structural integrity checks
# ============================================================
_config_text_after = BP6_CONFIG_PATH.read_text(encoding="utf-8")
_front_matter_and_gate2_preserved = all(
    marker in _config_text_after
    for marker in ('bp_id: "bp6"', GATE2_MARKER_TEXT, "pii_screen_rows_scanned:", "random_state: 42")
)

_checks: list[tuple[str, bool]] = [
    ("both_real_strategies_evaluated", len(champion_result["candidates_evaluated"]) == 2),
    ("champion_is_taxonomy_bucket_match", champion_is_taxonomy_bucket_match),
    ("champion_coverage_strictly_greater_than_runner_up", champion_result["champion_coverage"] > champion_result["runner_up_coverage"]),
    ("strategy_a_coverage_in_valid_range", 0.0 <= strategy_a["coverage"] <= 1.0),
    ("strategy_b_coverage_in_valid_range", 0.0 <= strategy_b["coverage"] <= 1.0),
    ("strategy_b_naive_baseline_legitimately_weak", strategy_b["coverage"] < strategy_a["coverage"]),
    ("retrieval_inventory_entry_written", RETRIEVAL_INVENTORY_PATH.exists()),
    ("zero_genai_sdk_modules_loaded", loaded_sdks == []),
    ("config_gate3_block_written", gate3_marker in _config_text_after),
    ("config_front_matter_and_gate2_block_preserved", _front_matter_and_gate2_preserved),
]

_failed = [name for name, ok in _checks if not ok]
for name, ok in _checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert not _failed, f"BP6 Gate 3 structural integrity checks failed: {_failed}"

print(
    "\n[ALL CHECKS PASSED] BP6 Gate 3 (Retrieval Strategy Benchmark & Champion Selection) "
    f"complete. Champion: {champion_result['champion_strategy']} "
    f"(coverage {champion_result['champion_coverage']:.4f}). This gate is prep-only - BP6's own "
    "retrieval/generation work begins at Gate 5."
)
